# 03. Topic Modeling

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

pd.set_option('display.max_colwidth', 100)

In [ ]:
try:
    df = pd.read_csv('../data/processed/reviews_sentiment.csv')
except FileNotFoundError:
    df = pd.read_csv('../data/processed/reviews_clean.csv')

df['token_str'] = df['token_str'].fillna('')

print(df.shape)
df[['token_str']].head(3)

## 1. Vocabulary Inspection

In [ ]:
from collections import Counter

all_tokens = []

for token_str in df['token_str']:
    all_tokens.extend(token_str.split())

counter = Counter(all_tokens)

print(f'Total tokens : {len(all_tokens):,}')
print(f'Unique tokens: {len(counter):,}')

print('\nTop 30 most common tokens:')

for word, count in counter.most_common(30):
    print(f'  {word:<20} {count:>8,}')

In [ ]:
DOMAIN_STOPWORDS = set([
    # 'product', 'order', 'amazon', 'purchase', 'get', 'use', 'would', 'could', 'also', 'item', 'buy',
])

if DOMAIN_STOPWORDS:

    df['token_str'] = df['token_str'].apply(
        lambda s: ' '.join(
            t for t in s.split()
            if t not in DOMAIN_STOPWORDS
        )
    )

    print('Applied domain stopwords')

else:
    print('No domain stopwords set - tokens unchanged')

## 2. Vectorizer Parameter Tuning

In [ ]:
for min_df in [2,5,10,20]:

    for max_df in [0.95,0.90,0.80]:

        vec = CountVectorizer(
            min_df=min_df,
            max_df=max_df,
            max_features=5000
        )

        dtm = vec.fit_transform(df['token_str'])

        print(
            f'min_df={min_df:<3} '
            f'max_df={max_df} '
            f'→ vocab size: {len(vec.vocabulary_):>5,} '
            f'| matrix: {dtm.shape}'
        )

In [ ]:
MIN_DF = 5
MAX_DF = 0.90
MAX_FEATURES = 5000

vectorizer = CountVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
    ngram_range=(1,2)
)

dtm = vectorizer.fit_transform(
    df['token_str']
)

print(
    f'DTM shape: {dtm.shape}'
)

## 3. Finding the Right Number of Topics

In [ ]:
# takes long to get executed completely
topic_range = range(3,15)
perplexities = []

for k in topic_range:
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method='batch',
        max_iter=10
    )
    lda.fit(dtm)
    p = lda.perplexity(dtm)
    perplexities.append(p)
    print(
        f'k={k} '
        f'perplexity={p:.1f}'
    )

In [ ]:
plt.plot(list(topic_range), perplexities, 'o-')
plt.title('LDA Perplexity vs Topics')
plt.xlabel('Topics')
plt.ylabel('Perplexity')
plt.show()

In [ ]:
N_TOPICS = 8

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method='batch',
    max_iter=20,
    doc_topic_prior=0.1,
    topic_word_prior=0.01
)
doc_topics = lda.fit_transform(dtm)
print(lda.perplexity(dtm))

## 4. Inspect Topics

In [ ]:
feature_names = (vectorizer.get_feature_names_out())
N_WORDS = 15

topic_data = []
for topic_idx, topic in enumerate(lda.components_):
    top_indices = (topic.argsort()[-N_WORDS:][::-1])
    words = [feature_names[i] for i in top_indices]
    topic_data.append({'topic':topic_idx+1,'words':words})
    print(f'Topic {topic_idx+1}: ' + " | ".join(map(str, words[:10])))

In [ ]:
TOPIC_LABELS = {
    1:'Battery Life',
    2:'Build Quality',
    3:'Sound / Audio',
    4:'Display / Screen',
    5:'Connectivity',
    6:'Price / Value',
    7:'Performance',
    8:'Customer Service'
}

## 5. Topic Distribution

In [ ]:
df['dominant_topic'] = (doc_topics.argmax(axis=1))
df['topic_confidence'] = (doc_topics.max(axis=1))
dist = (df['dominant_topic'].value_counts().sort_index())
labels = [TOPIC_LABELS.get(i+1,f'Topic {i+1}') for i in dist.index]
dist.plot(kind='barh')
plt.yticks(range(len(labels)), labels)
plt.show()

In [ ]:
INSPECT_TOPIC = 0
topic_reviews = df[df['dominant_topic'] == INSPECT_TOPIC].sort_values('topic_confidence', ascending=False)
for _, row in (topic_reviews.head(5).iterrows()):
    print(f'[{row["topic_confidence"]:.3f}] ', f'{str(row["clean_text"])[:120]}')

## 6. TF-IDF Keyword Extraction

In [ ]:
tfidf = TfidfVectorizer(max_features=1000,min_df=5,max_df=0.85,ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['token_str'])
scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten() # type: ignore
feature_names_tfidf = (tfidf.get_feature_names_out())
kw_df = pd.DataFrame({'keyword':feature_names_tfidf, 'score':scores})
kw_df = (kw_df.sort_values('score',ascending=False).head(30))
print(kw_df)

In [ ]:
if 'predicted_sentiment' in df.columns:
    for sentiment in ['positive', 'negative']:
        subset = df[df['predicted_sentiment'] == sentiment]
        if len(subset) < 50:
            continue
        tfidf_sub = TfidfVectorizer(max_features=500,min_df=3,max_df=0.90,ngram_range=(1,2))
        matrix = tfidf_sub.fit_transform(subset['token_str'])
        scores_sub = np.asarray(matrix.mean(axis=0)).flatten() # type: ignore
        top_kw = sorted(zip(tfidf_sub.get_feature_names_out(), scores_sub),key=lambda x: -x[1])[:15]
        print(f'\n{sentiment.upper()}')
        for word,score in top_kw:
            print( f'{word:<25} {score:.6f}')